In [1]:
import gurobipy as gp
from gurobipy import GRB


import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Input

In [2]:
import json

with open("data/toy_instance.json") as f:
    inst = json.load(f)

# Sets
h = inst["horizon"]
H = range(1, h + 1)                              # time horizon {1, ..., h}
Q = inst["qualifications"]                       # qualifications
S = [s["name"] for s in inst["staff"]]           # staff
J = [j["name"] for j in inst["jobs"]]            # projects

# Staff parameters
Q_i = {s["name"]: set(s["qualifications"]) for s in inst["staff"]}   # Q_i ⊆ Q
V_i = {s["name"]: set(s["vacations"]) for s in inst["staff"]}        # V_i ⊆ H

# Project parameters
Q_j = {j["name"]: set(j["working_days_per_qualification"]) for j in inst["jobs"]}   # Q_j' ⊆ Q
n = {(j["name"], k): days
     for j in inst["jobs"]
     for k, days in j["working_days_per_qualification"].items()}     # n[j, k]
g = {j["name"]: j["gain"] for j in inst["jobs"]}                     # gain g_j
d = {j["name"]: j["due_date"] for j in inst["jobs"]}                 # due date d_j
c = {j["name"]: j["daily_penalty"] for j in inst["jobs"]}            # penalty c_j'

In [3]:
n

{('Job1', 'A'): 1,
 ('Job1', 'B'): 1,
 ('Job1', 'C'): 1,
 ('Job2', 'A'): 1,
 ('Job2', 'B'): 2,
 ('Job3', 'A'): 1,
 ('Job3', 'C'): 2,
 ('Job4', 'B'): 2,
 ('Job4', 'C'): 1,
 ('Job5', 'C'): 2}

In [4]:
V_i

{'Olivia': set(), 'Liam': {1}, 'Emma': {2}}

In [5]:
Q_j

{'Job1': {'A', 'B', 'C'},
 'Job2': {'A', 'B'},
 'Job3': {'A', 'C'},
 'Job4': {'B', 'C'},
 'Job5': {'C'}}

In [6]:
for k in Q_j["Job1"]:
    print(k)

B
C
A


In [7]:
for i in n:
    print(i)

('Job1', 'A')
('Job1', 'B')
('Job1', 'C')
('Job2', 'A')
('Job2', 'B')
('Job3', 'A')
('Job3', 'C')
('Job4', 'B')
('Job4', 'C')
('Job5', 'C')


In [8]:
g

{'Job1': 20, 'Job2': 15, 'Job3': 15, 'Job4': 20, 'Job5': 10}

In [9]:
d

{'Job1': 3, 'Job2': 3, 'Job3': 4, 'Job4': 3, 'Job5': 5}

In [10]:
c

{'Job1': 3, 'Job2': 3, 'Job3': 3, 'Job4': 3, 'Job5': 3}

# Model

In [11]:
# Create initial model.
m = gp.Model("staff")

Restricted license - for non-production use only - expires 2027-11-29


# Variables

In [12]:
# Initialize assignment decision variables.

# x[i,t,j,k]: only valid combinations (qualified, not on vacation, project needs k)
idx = [(i, t, j, k)
       for i in S
       for t in H if t not in V_i[i]
       for j in J
       for k in Q_j[j] & Q_i[i]]

idx

[('Olivia', 1, 'Job1', 'B'),
 ('Olivia', 1, 'Job1', 'C'),
 ('Olivia', 1, 'Job1', 'A'),
 ('Olivia', 1, 'Job2', 'B'),
 ('Olivia', 1, 'Job2', 'A'),
 ('Olivia', 1, 'Job3', 'C'),
 ('Olivia', 1, 'Job3', 'A'),
 ('Olivia', 1, 'Job4', 'B'),
 ('Olivia', 1, 'Job4', 'C'),
 ('Olivia', 1, 'Job5', 'C'),
 ('Olivia', 2, 'Job1', 'B'),
 ('Olivia', 2, 'Job1', 'C'),
 ('Olivia', 2, 'Job1', 'A'),
 ('Olivia', 2, 'Job2', 'B'),
 ('Olivia', 2, 'Job2', 'A'),
 ('Olivia', 2, 'Job3', 'C'),
 ('Olivia', 2, 'Job3', 'A'),
 ('Olivia', 2, 'Job4', 'B'),
 ('Olivia', 2, 'Job4', 'C'),
 ('Olivia', 2, 'Job5', 'C'),
 ('Olivia', 3, 'Job1', 'B'),
 ('Olivia', 3, 'Job1', 'C'),
 ('Olivia', 3, 'Job1', 'A'),
 ('Olivia', 3, 'Job2', 'B'),
 ('Olivia', 3, 'Job2', 'A'),
 ('Olivia', 3, 'Job3', 'C'),
 ('Olivia', 3, 'Job3', 'A'),
 ('Olivia', 3, 'Job4', 'B'),
 ('Olivia', 3, 'Job4', 'C'),
 ('Olivia', 3, 'Job5', 'C'),
 ('Olivia', 4, 'Job1', 'B'),
 ('Olivia', 4, 'Job1', 'C'),
 ('Olivia', 4, 'Job1', 'A'),
 ('Olivia', 4, 'Job2', 'B'),
 ('Olivia', 4,

In [13]:
# matrix of peapole, job, skill, day
x = m.addVars(idx, vtype=GRB.BINARY, name="x")

# auxilliary variable: project j completed in horizon
y = m.addVars(J, vtype=GRB.BINARY, name='y')

# completion day of the project
e = m.addVars(J, vtype=GRB.INTEGER, lb=0, ub=h, name='e')

# number of days late of the project
l = m.addVars(J, vtype=GRB.INTEGER, lb=0, name='l')

# Constraints

In [14]:
# one staff member works on at most one project (and one skill) per day
daily_assignment = m.addConstrs(
    (x.sum(i, t, "*", "*") <= 1 for i in S for t in H if t not in V_i[i]),
    name="daily_assignment",
)
daily_assignment

{('Olivia', 1): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 2): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 3): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 4): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 5): <gurobi.Constr *Awaiting Model Update*>,
 ('Liam', 2): <gurobi.Constr *Awaiting Model Update*>,
 ('Liam', 3): <gurobi.Constr *Awaiting Model Update*>,
 ('Liam', 4): <gurobi.Constr *Awaiting Model Update*>,
 ('Liam', 5): <gurobi.Constr *Awaiting Model Update*>,
 ('Emma', 1): <gurobi.Constr *Awaiting Model Update*>,
 ('Emma', 3): <gurobi.Constr *Awaiting Model Update*>,
 ('Emma', 4): <gurobi.Constr *Awaiting Model Update*>,
 ('Emma', 5): <gurobi.Constr *Awaiting Model Update*>}

In [15]:
# a project is either fully covered for each qualification (y=1) or not worked on at all (y=0)
completed_project = m.addConstrs((x.sum("*", "*", j, k) == n[j,k] * y[j] for j in J for k in Q_j[j]), name="coverage",)
completed_project

{('Job1', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job1', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job1', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job2', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job2', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job3', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job3', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job4', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job4', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Job5', 'C'): <gurobi.Constr *Awaiting Model Update*>}

In [16]:
# the completion day is at least every day someone works on the project
end_day = m.addConstrs((e[j] >= t * x[i, t, j, k] for (i, t, j, k) in idx), name="end_day")
end_day

{('Olivia', 1, 'Job1', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job1', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job1', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job2', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job2', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job3', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job3', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job4', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job4', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 1, 'Job5', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 2, 'Job1', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 2, 'Job1', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 2, 'Job1', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 2, 'Job2', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('Olivia', 2, 'Job2

In [17]:
# days late = completion day - due date (0 if on time)
lateness = m.addConstrs((l[j] >= e[j] - d[j] for j in J), name="lateness")
lateness

{'Job1': <gurobi.Constr *Awaiting Model Update*>,
 'Job2': <gurobi.Constr *Awaiting Model Update*>,
 'Job3': <gurobi.Constr *Awaiting Model Update*>,
 'Job4': <gurobi.Constr *Awaiting Model Update*>,
 'Job5': <gurobi.Constr *Awaiting Model Update*>}

# Objective functions

In [18]:
# Project brief specifies:
# 1) Maximize total profit realized
# 2) Minimize the actual project execution time
# 3) Minimize the number of projects an employee works on
# 4) Minimize the number of projects delivered late

def objective_profit(y,g,c,l):
    # sum over projects:
    # gain(g)*complete(y) - late penalty(c)*days late(l)
    return gp.quicksum(
        g[j]*y[j] - c[j]*l[j] for j in J
    )

def objective_execution_time(m,e):
    # last completion day over all projects
    M = m.addVar(name="last_day")
    m.addConstr(M == gp.max_(list(e.values())))
    return M

def objective_projects_per_employee(m,x):
    # sum over projects and employees worked

    # first check if employee 'i' works on project 'j'
    z = m.addVars(S, J, vtype=GRB.BINARY, name="project_worked")
    for i in S:
        for j in J:
            m.addGenConstrOr( # this sets z[i,j] to 1 if any of the x[..] are 1s for any day/project
                z[i,j],
                [x[i,t,j,k] for t in H for k in Q if (i,t,j,k) in x]
            )

    # define a new variable that will hold the max no. projects worked on by any employee
    P = m.addVar(name = "max_projects")

    # which is the result of constraints on being at least as large as # projects for each employee
    for i in S:
        m.addConstr(P >= gp.quicksum(z[i,j] for j in J))

    return P

def objective_projects_late(m,l):
    # return no. of projects that are late

    # define variable r[j] to hold 0 if not late, 1 if late
    r = m.addVars(J, vtype=GRB.BINARY, name="late")

    # the constraints are = 0 if l[j] is zero, =1 if l[j] is positive (l is integer so > 0 <=> >= 1)
    for j in J:
        m.addGenConstrIndicator(r[j], 0, l[j] == 0)
        m.addGenConstrIndicator(r[j], 1, l[j] >= 1)

    return gp.quicksum(r[j] for j in J)

In [19]:
# build the 4 objectives
profit = objective_profit(y, g, c, l)
execution_time = objective_execution_time(m, e)
projects_per_employee = objective_projects_per_employee(m, x)
projects_late = objective_projects_late(m, l)

In [20]:
# lexicographic (hierarchical) optimization, in the order of the brief:
# the higher the priority, the earlier the objective is optimized.
# ModelSense is MAXIMIZE, so the objectives to minimize get weight = -1
m.ModelSense = GRB.MAXIMIZE

m.setObjectiveN(profit,                index=0, priority=4, weight=1,  name="profit")
m.setObjectiveN(execution_time,        index=1, priority=3, weight=-1, name="execution_time")
m.setObjectiveN(projects_per_employee, index=2, priority=2, weight=-1, name="projects_per_employee")
m.setObjectiveN(projects_late,         index=3, priority=1, weight=-1, name="projects_late")

# Solve

In [21]:
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 27.0.0 26A434)


CPU model: Apple M5 Pro


Thread count: 18 physical cores, 18 logical processors, using up to 18 threads


Optimize a model with 121 rows, 127 columns and 398 nonzeros (Max)


Model fingerprint: 0xac251bc1


Model has 10 linear objective coefficients


... (101 outputs left out)

Optimal solution found (tolerance 1.00e-04)


Best objective -0.000000000000e+00, best bound -0.000000000000e+00, gap 0.0000%


---------------------------------------------------------------------------


Multi-objectives: solved in 0.04 seconds (0.02 work units), solution count 7


# Results

In [22]:
# value of each objective
for o in range(m.NumObj):
    m.Params.ObjNumber = o
    print(f"{m.ObjNName}: {m.ObjNVal:g}")

profit: 65
execution_time: 5
projects_per_employee: 2
projects_late: 0


In [23]:
# schedule: staff x days
schedule = pd.DataFrame("", index=S, columns=list(H))
for i in S:
    for t in V_i[i]:
        schedule.loc[i, t] = "vacation"
for (i, t, j, k), var in x.items():
    if var.X > 0.5:
        schedule.loc[i, t] = f"{j} ({k})"

schedule

,1,2,3,4,5
Olivia,Job4 (B),Job4 (B),Job4 (C),Job5 (C),Job5 (C)
Liam,vacation,Job1 (B),Job1 (A),Job3 (A),
Emma,Job1 (C),vacation,Job3 (C),Job3 (C),


In [24]:
# projects: completed or not, completion day and delay
projects = pd.DataFrame({
    "completed": {j: int(round(y[j].X)) for j in J},
    "end_day":   {j: int(round(e[j].X)) if y[j].X > 0.5 else None for j in J},
    "due_date":  d,
    "days_late": {j: int(round(l[j].X)) for j in J},
    "gain":      g,
})

projects

,completed,end_day,due_date,days_late,gain
Job1,1,3.0,3,0,20
Job2,0,NaN,3,0,15
Job3,1,4.0,4,0,15
Job4,1,3.0,3,0,20
Job5,1,5.0,5,0,10
